In [ ]:
# 05_evaluation.ipynb
# Paper + Artifact Evaluation (EVALUATION ONLY)
"""
This notebook:
- Loads saved model logits
- Computes Binary F1 and Macro F1 for all 6 models
- Saves evaluation plots to: ../plots/evaluation/

Key Features:
- Paper-compliant 18-class gesture evaluation
- Binary (BFRB vs non-BFRB) and Macro F1 metrics
- Visualizations for model comparison
"""
# Standard imports
from pathlib import Path
import json
import numpy as np
from collections import Counter
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score, confusion_matrix


In [ ]:

# --- Helper: Find the repo root ---
def find_repo_root(start: Path | None = None) -> Path:
    """Walk up the directory tree until we find .git or README.md."""
    p = (start or Path.cwd()).resolve()
    for _ in range(10):
        if (p / ".git").exists() or (p / "README.md").exists():
            return p
        p = p.parent
    return (start or Path.cwd()).resolve()

# Set up paths
REPO_ROOT = find_repo_root()
DATA_DIR     = REPO_ROOT / "data" / "processed" / "cmi_sensor_data"
ARTIFACT_DIR = REPO_ROOT / "models_artifacts"
OUTPUT_DIR   = ARTIFACT_DIR / "outputs"
METADATA_DIR = ARTIFACT_DIR / "metadata"
PLOTS_DIR    = REPO_ROOT / "plots" / "evaluation"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

print("REPO_ROOT   :", REPO_ROOT)
print("DATA_DIR    :", DATA_DIR)
print("OUTPUT_DIR  :", OUTPUT_DIR)
print("METADATA_DIR:", METADATA_DIR)
print("PLOTS_DIR   :", PLOTS_DIR)

In [ ]:

# --- Check for required files ---
REQUIRED_FILES = [
    METADATA_DIR / "class_mapping.json",
    OUTPUT_DIR / "val_seq_ids.npy",
    OUTPUT_DIR / "logits_val_fft_mlp_all.npy",
    OUTPUT_DIR / "logits_val_fft_mlp_imu_thm.npy",
    OUTPUT_DIR / "logits_val_fft_rf_all.npy",
    OUTPUT_DIR / "logits_val_cnn_bilstm_tof.npy",
    OUTPUT_DIR / "logits_val_late_fusion.npy",
    OUTPUT_DIR / "logits_val_intermediate_fusion.npy",
    DATA_DIR / "train_clean.csv",
]

missing = [str(p) for p in REQUIRED_FILES if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Evaluation cannot proceed. Missing required files:\n" +
        "\n".join([f"  - {m}" for m in missing])
    )
print("✅ All required files found!")

In [ ]:

# --- Load class mapping (paper: 18 gesture classes) ---
with open(METADATA_DIR / "class_mapping.json", "r", encoding="utf-8") as f:
    class_mapping = json.load(f)

CLASSES = [str(c).strip() for c in class_mapping["classes"]]
NUM_CLASSES = len(CLASSES)
assert NUM_CLASSES == 18, f"Paper requires 18 output variables, got {NUM_CLASSES}"
print("✅ Loaded class mapping; NUM_CLASSES =", NUM_CLASSES)

# Paper-defined 8 BFRB gestures (must match names in class_mapping)
PAPER_BFRB_GESTURES = {
    "Neck - scratch",
    "Eyebrow - pull hair",
    "Forehead - scratch",
    "Forehead - pull hairline",
    "Above ear - pull hair",
    "Neck - pinch skin",
    "Eyelash - pull hair",
    "Cheek - pinch skin",
}
BFRB_SET = {g.strip() for g in PAPER_BFRB_GESTURES}

bfrb_idx = [i for i, g in enumerate(CLASSES) if g in BFRB_SET]
assert len(bfrb_idx) == 8, (
    f"Expected 8 BFRB gestures in CLASSES; got {len(bfrb_idx)}.\n"
    f"CLASSES: {CLASSES}\nBFRB_SET: {sorted(BFRB_SET)}"
)
print(" BFRB classes used:", [CLASSES[i] for i in bfrb_idx])

In [ ]:

# --- Load val ids and recompute ground truth ---
val_seq_ids = np.load(OUTPUT_DIR / "val_seq_ids.npy", allow_pickle=True)
N = len(val_seq_ids)
assert N > 0, "val_seq_ids is empty"

train_clean = pd.read_csv(DATA_DIR / "train_clean.csv")
SEQ_COL = "sequence_id"
GESTURE_COL = "gesture"

if SEQ_COL not in train_clean.columns or GESTURE_COL not in train_clean.columns:
    raise KeyError(f"train_clean.csv must contain '{SEQ_COL}' and '{GESTURE_COL}'")

val_rows = train_clean[train_clean[SEQ_COL].isin(val_seq_ids)].copy()
if val_rows.empty:
    raise ValueError("No rows in train_clean match val_seq_ids. Check split generation and ids type.")

# Ensure deterministic per-sequence label: take first row per sequence after sorting by sequence_id.
val_first = (
    val_rows.sort_values([SEQ_COL])
    .groupby(SEQ_COL, as_index=False)
    .first()
)

# Align order to val_seq_ids (so y_true matches logits order)
val_first_indexed = val_first.set_index(SEQ_COL)
try:
    gesture_str = val_first_indexed.loc[val_seq_ids, GESTURE_COL].astype(str).str.strip().to_numpy()
except KeyError:
    missing_ids = set(val_seq_ids.tolist()) - set(val_first_indexed.index.tolist())
    raise KeyError(f"Some val_seq_ids are missing in train_clean after preprocessing: {list(sorted(missing_ids))[:10]}")

# Paper macro truth: BFRB gestures keep identity; others -> non_target
y_true_macro = np.where(np.isin(gesture_str, list(BFRB_SET)), gesture_str, "non_target").astype(object)
# Paper binary truth: BFRB=1, non_target=0
y_true_binary = (y_true_macro != "non_target").astype(int)

assert len(y_true_macro) == N and len(y_true_binary) == N
print("DEBUG y_true_binary distribution:", Counter(y_true_binary.tolist()))
print("DEBUG y_true_macro top10:", Counter(map(str, y_true_macro)).most_common(10))
assert 0 in set(y_true_binary.tolist()) and 1 in set(y_true_binary.tolist()), \
    "Ground truth collapsed (all 0 or all 1). Cannot compute binary F1."


In [ ]:

# --- Load logits (paper models) ---
MODEL_LOGITS = {
    "FFT-MLP (All)"        : np.load(OUTPUT_DIR / "logits_val_fft_mlp_all.npy"),
    "FFT-MLP (IMU+THM)"    : np.load(OUTPUT_DIR / "logits_val_fft_mlp_imu_thm.npy"),
    "FFT-RF (All)"         : np.load(OUTPUT_DIR / "logits_val_fft_rf_all.npy"),
    "CNN-BiLSTM (TOF)"     : np.load(OUTPUT_DIR / "logits_val_cnn_bilstm_tof.npy"),
    "Late Fusion"          : np.load(OUTPUT_DIR / "logits_val_late_fusion.npy"),
    "Intermediate Fusion"  : np.load(OUTPUT_DIR / "logits_val_intermediate_fusion.npy"),
}

# Optional: FFT-MLP (IMU) required for Paper Fig. 6.
imu_logits_path = OUTPUT_DIR / "logits_val_fft_mlp_imu.npy"
if imu_logits_path.exists():
    MODEL_LOGITS["FFT-MLP (IMU)"] = np.load(imu_logits_path)

# Validate shapes
for name, arr in MODEL_LOGITS.items():
    assert arr.ndim == 2 and arr.shape[0] == N and arr.shape[1] == 18, \
        f"{name}: expected shape (N,18)=({N},18), got {arr.shape}"
print("✅ All logits loaded and validated:", list(MODEL_LOGITS.keys()))

In [ ]:

# --- Robust logits -> probabilities ---
def logits_to_probs(arr: np.ndarray) -> np.ndarray:
    """
    Converts raw logits to probabilities using numerically stable softmax.
    If the values look like log-probabilities, this still produces valid probs.
    """
    arr = np.asarray(arr, dtype=np.float64)
    arr = arr - np.max(arr, axis=1, keepdims=True)
    exp = np.exp(arr)
    return exp / np.maximum(exp.sum(axis=1, keepdims=True), 1e-12)

In [ ]:

# --- Compute metrics (paper-compliant) ---
EVALUATION_RESULTS = {}
PREDICTION_STORE = {}

for model_name, logits in MODEL_LOGITS.items():
    probs18 = logits_to_probs(logits)

    # Sanity: probabilities sum to 1
    row_sums = probs18.sum(axis=1)
    if not np.allclose(row_sums, 1.0, atol=1e-6):
        raise ValueError(f"{model_name}: probs do not sum to 1 (min={row_sums.min()}, max={row_sums.max()})")

    # Binary prediction per paper
    p_bfrb = probs18[:, bfrb_idx].sum(axis=1)
    y_pred_bin = (p_bfrb >= 0.5).astype(int)

    # Macro prediction per paper: argmax gesture then collapse non-BFRB -> non_target
    pred_idx = probs18.argmax(axis=1)
    gesture_pred = np.array([CLASSES[i] for i in pred_idx], dtype=object)
    y_pred_macro = np.where(np.isin(gesture_pred, list(BFRB_SET)), gesture_pred, "non_target").astype(object)

    # F1s
    bin_f1 = f1_score(y_true_binary, y_pred_bin)
    mac_f1 = f1_score(y_true_macro, y_pred_macro, average="macro")

    EVALUATION_RESULTS[model_name] = {"binary_f1": float(bin_f1), "macro_f1": float(mac_f1)}
    PREDICTION_STORE[model_name] = {
        "binary": y_pred_bin,
        "macro": y_pred_macro,
        "gesture": gesture_pred,
        "p_bfrb": p_bfrb,
    }

print("✅ Metrics computed")
print("DEBUG binary_f1:", {m: EVALUATION_RESULTS[m]["binary_f1"] for m in EVALUATION_RESULTS})

In [ ]:

# --- Paper-style plots (Figures 4–7) + extra confusion matrix ---
def save_bar_plot(fig_name, labels, values_pct, ylim, ylabel, title):
    print("Plotting:", dict(zip(labels, values_pct)))  # add this line

    plt.figure(figsize=(8, 4.5))
    plt.bar(labels, values_pct)
    plt.ylabel(ylabel)
    plt.ylim(*ylim)
    plt.title(title, pad=10)
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    out = PLOTS_DIR / fig_name
    plt.savefig(out, dpi=200)
    plt.close()
    print("Saved:", out.resolve())


In [ ]:

# Figure 4: Binary F1 models trained on all sensor data
FIG4 = ["FFT-MLP (All)", "FFT-RF (All)", "Intermediate Fusion", "Late Fusion"]
vals4 = [EVALUATION_RESULTS[m]["binary_f1"] * 100 for m in FIG4]
save_bar_plot(
    "fig4_binary_f1_all_inputs.png",
    FIG4,
    vals4,
    (80, 100),
    "Binary F1 (%)",
    "Binary F1-scores (All Sensor Inputs)"
)

# Figure 5: Macro F1 models trained on all sensor data (8 BFRBs + non_target)
vals5 = [EVALUATION_RESULTS[m]["macro_f1"] * 100 for m in FIG4]
save_bar_plot(
    "fig5_macro_f1_all_inputs.png",
    FIG4,
    vals5,
    (0, 70),
    "Macro F1 (%)",
    "Macro F1-scores (BFRB Types; non-BFRB→non_target)"
)

# Figure 6: FFT-MLP variants (requires FFT-MLP(IMU) logits to fully match paper)
if "FFT-MLP (IMU)" in EVALUATION_RESULTS:
    FIG6 = ["FFT-MLP (All)", "FFT-MLP (IMU+THM)", "FFT-MLP (IMU)"]
    vals6 = [EVALUATION_RESULTS[m]["binary_f1"] * 100 for m in FIG6]
    save_bar_plot(
        "fig6_binary_f1_fft_mlp_variants.png",
        FIG6,
        vals6,
        (84, 100),
        "Binary F1 (%)",
        "Binary F1-scores (FFT-MLP Input Variants)"
    )
else:
    print("NOTE: logits_val_fft_mlp_imu.npy not found; skipping Fig. 6 (FFT-MLP IMU-only).")

# Figure 7: Constituents vs ensembles
FIG7 = ["FFT-MLP (IMU+THM)", "CNN-BiLSTM (TOF)", "Late Fusion", "Intermediate Fusion"]
vals7 = [EVALUATION_RESULTS[m]["binary_f1"] * 100 for m in FIG7]
save_bar_plot(
    "fig7_binary_f1_constituents_vs_ensembles.png",
    FIG7,
    vals7,
    (84, 100),
    "Binary F1 (%)",
    "Binary F1-scores (Constituents vs Ensembles)"
)


In [ ]:

# Extra: Confusion matrix for best macro-F1 model
best_model = max(EVALUATION_RESULTS.keys(), key=lambda k: EVALUATION_RESULTS[k]["macro_f1"])
y_pred_best = PREDICTION_STORE[best_model]["macro"]

macro_labels = sorted(
    set(y_true_macro.astype(str).tolist() +
        y_pred_best.astype(str).tolist())
)
cmat = confusion_matrix(y_true_macro, y_pred_best, labels=macro_labels)

plt.figure(figsize=(12, 10))
plt.imshow(cmat, cmap="Blues", interpolation="nearest")
plt.title(f"Confusion Matrix — {best_model} (Macro F1={EVALUATION_RESULTS[best_model]['macro_f1']:.3f})", pad=12)
plt.colorbar()
ticks = np.arange(len(macro_labels))
plt.xticks(ticks, macro_labels, rotation=45, ha="right")
plt.yticks(ticks, macro_labels)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "confusion_matrix_best_macro_model.png", dpi=200)
plt.close()
print("Saved: confusion_matrix_best_macro_model.png")

In [ ]:

# --- Display plots ---
from IPython.display import Image

Image(PLOTS_DIR / "fig4_binary_f1_all_inputs.png")
Image(PLOTS_DIR / "fig5_macro_f1_all_inputs.png")
Image(PLOTS_DIR / "fig7_binary_f1_constituents_vs_ensembles.png")

if "FFT-MLP (IMU)" in EVALUATION_RESULTS:
    Image(PLOTS_DIR / "fig6_binary_f1_fft_mlp_variants.png")
Image(PLOTS_DIR / "confusion_matrix_best_macro_model.png")


In [ ]:

# --- Structural + artifact plot checks ---
pngs = sorted(PLOTS_DIR.glob("*.png"))
print("Found evaluation plots:", [p.name for p in pngs])
assert len(pngs) >= 3, f"Need >= 3 plots in {PLOTS_DIR}, found {len(pngs)}."

# Non-empty plot sanity (avoid blank files)
for p in pngs:
    assert p.stat().st_size > 10_000, f"Plot {p.name} looks too small; may be blank/corrupt."

print("✅ Evaluation complete and artifact-compliant.")


